# 州の平均注文額の差をカテゴリ別に分ける

## なぜこの計算をするか

PB・APの平均注文額はSPより高い。州の平均注文額を各カテゴリの金額に分けると、どのカテゴリの過去の売上構成に差があるかを確認できる。

**カテゴリの平均注文額への構成額 = カテゴリGMV ÷ 州のユニークな注文数**。全カテゴリの構成額を足すと、その州の平均注文額になる。PB・APの構成額からSPの構成額を引いた差を並べる。

この差は観察された注文構成の差であり、カテゴリが注文額を増やした因果効果や販促時の増分GMVではない。少数の高額注文にも注意する。

In [ ]:
from pathlib import Path
import sys

# Notebookをどの作業フォルダから開いてもsrc/を読み込めるようにする。
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.project_paths import DATA_DIR
from src.sales_metrics import state_totals as build_state_totals
from src.sales_metrics import category_state_metrics, state_aov_difference_by_category

df_baseline = pd.read_csv(DATA_DIR / "sales_baseline_by_month_state.csv")
df_category = pd.read_csv(DATA_DIR / "sales_by_category_state.csv")
state_totals = build_state_totals(df_baseline)
category_parts = category_state_metrics(df_category, df_baseline)
display(state_totals.loc[
    state_totals["customer_state"].isin(["SP", "PB", "AP"]),
    ["customer_state", "state_orders", "average_order_value"],
].round(2))

In [ ]:
# 州AOV差 = カテゴリ別構成額の差の合計。少数注文カテゴリは件数も見る。
for state in ["PB", "AP"]:
    comparison = state_aov_difference_by_category(category_parts, state, "SP")
    print(f"{state}とSPの平均注文額差: {comparison['aov_part_difference'].sum():.2f}")
    display(comparison.nlargest(10, "aov_part_difference").round(2))

## 結果をどう読むか

- 差が大きいカテゴリでは、対象州の件数も確認する。APのように州全体の注文数が少ない場合、数件で構成額が大きく変わる。
- この表は州間の平均注文額差を分けたもの。購入割合の差とカテゴリ内の金額差をさらに区別するには、追加の計算が必要。
- 販促対象の選定には、期間ごとの安定性、商品・販売者の供給、広告費と利益率なども必要。次の問いは表を見てから選ぶ。